In [ ]:
# Run this cell BEFORE saving to GitHub
import nbformat
import json

# Read current notebook
with open('/content/drive/MyDrive/your_notebook_name.ipynb', 'r') as f:
    nb = json.load(f)

# Remove problematic widget metadata
if 'widgets' in nb.get('metadata', {}):
    del nb['metadata']['widgets']

# Save cleaned version
with open('/content/drive/MyDrive/your_notebook_name.ipynb', 'w') as f:
    json.dump(nb, f, indent=1)

print("Done! Now save this to GitHub.")

In [ ]:
!pip install transformers sentencepiece sacremoses -q

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import easyocr
import whisper
import gradio as gr

# --- OCR ---
print("Loading OCR...")
reader = easyocr.Reader(['en'])

# --- Translation (direct, no pipeline) ---
print("Loading NLLB translation model...")
nllb_model_name = "facebook/nllb-200-distilled-600M"
nllb_tokenizer = AutoTokenizer.from_pretrained(nllb_model_name)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_model_name)
print("✅ Translation model loaded!")

# --- Whisper ---
print("Loading Whisper...")
voice_model = whisper.load_model("small")
print("✅ All models loaded!")

# --- Functions ---
def translate_text(text):
    if not text.strip(): return ""
    inputs = nllb_tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )
    target_lang_id = nllb_tokenizer.convert_tokens_to_ids("urd_Arab")
    translated_tokens = nllb_model.generate(
        **inputs,
        forced_bos_token_id=target_lang_id,
        max_length=512
    )
    return nllb_tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

def translate_image(img):
    results = reader.readtext(img)
    extracted = " ".join([r[1] for r in results])
    return extracted, translate_text(extracted)

def translate_voice(audio_path):
    result = voice_model.transcribe(audio_path, language="en")
    transcribed = result['text']
    translated = translate_text(transcribed)
    return f"📝 Original: {transcribed}\n\n🔤 Urdu: {translated}"

# --- Gradio UI ---
with gr.Blocks(title="Urdu Translator") as demo:
    gr.Markdown("# 🇵🇰 AI English → Urdu Translator")

    with gr.Tab("📝 Text"):
        t_in = gr.Textbox(label="English Text", lines=3, placeholder="Type English here...")
        t_out = gr.Textbox(label="Urdu Translation", lines=3)
        gr.Button("Translate").click(translate_text, t_in, t_out)

    with gr.Tab("🖼️ Image"):
        i_in = gr.Image(type="filepath", label="Upload Image")
        with gr.Row():
            i_extracted = gr.Textbox(label="Extracted Text")
            i_out = gr.Textbox(label="Urdu Translation")
        gr.Button("Extract & Translate").click(
            translate_image, i_in, [i_extracted, i_out]
        )

    with gr.Tab("🎙️ Voice"):
        a_in = gr.Audio(type="filepath", label="Record or Upload Audio")
        a_out = gr.Textbox(label="Result", lines=4)
        gr.Button("Transcribe & Translate").click(translate_voice, a_in, a_out)

demo.launch(share=True, debug=True)

Loading OCR...
Loading NLLB translation model...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Translation model loaded!
Loading Whisper...


100%|███████████████████████████████████████| 461M/461M [00:06<00:00, 70.0MiB/s]


✅ All models loaded!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bdefc60da9c6d37107.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
